# Spark DataFrames on Databricks Free Edition

This notebook teaches the Spark DataFrame API in six sessions of about one hour each.
It is the lab notebook "Spark DataFrame API", moved to Databricks Free Edition.
API stands for application programming interface: the set of commands a library offers.

**Before your first session**
1. Run `00-setup-data.ipynb` once. It copies the course data into your workspace.
2. At the top right of this notebook, connect to **Serverless** compute.

**How to work through it**
- Run one cell at a time with **Shift+Enter**, from top to bottom.
- Read the text cell above each code cell before you run it.
- Some cells are **broken on purpose**. Their first line is `# EXPECT-ERROR:` and the name of the error.
  Run them, read the error, and then run the "Fix it" cell below them.
- Avoid **Run all** here. Databricks stops at the first error, and a broken cell comes early.
- Each session starts with a cell that loads everything that session needs.
  After a break, start again from that cell.

**Problems**
- Graded: Problems 2 and 3 (Session 5) and Problem 5 (Session 6).
- Optional stretch tasks: Problems 4, 6, 7, 8 and 9, at the end of the notebook.
- A cell that ends with `raise NotImplementedError()` is yours to complete.
  The test cell after it checks your work. It fails until your function is done.

**Contents**
- Session 1: Meet Spark and your first DataFrame
- Session 2: What's inside? Schemas, types and inspection
- Session 3: Imports and method chains
- Session 4: Debugging, when Spark complains
- Session 5: Parquet, files and the first graded problems
- Session 6: Nested JSON, text files and the capstone
- Optional stretch tasks

# Session 1: Meet Spark and your first DataFrame

**Goal.** By the end of this session you can start Spark, read the taxi data, and look at it.

**The analogy: a restaurant kitchen.**
- Your notebook places the orders.
- The **driver** is the head chef. It plans the work and splits it into small tasks.
- The **executors** are the line cooks. Each one works on its own slice of the data, at the same time as the others.
- On Databricks, the kitchen is already staffed and open. You do not hire cooks or light the stoves.
- `spark` is the waiter. It carries your orders to the kitchen and brings back the results.
  On Databricks the kitchen is in another building, and the waiter walks over the network.
  This way of working is called **Spark Connect**.

**Before you run**
- You ran `00-setup-data.ipynb` once, and its last cell printed only `OK` lines.
- This notebook is connected to Serverless compute.

## Notebook basics

A **notebook** is a page made of **cells**. A code cell holds Python. A text cell, like this one, holds notes.
Run a code cell with **Shift+Enter**. Its output appears under it.

A **variable** is a name that points to a value, such as `BASE` in the next cell.
Python knows a variable only after the cell that creates it has run.
So the **run order** matters: the order in which you ran cells, not their order on the page.

In [ ]:
# BASE is the folder (a Databricks "volume") that holds the course data.
BASE = "/Volumes/workspace/default/bdcc"

## What is Spark?

Apache Spark is a system for processing data that is too big for one computer.
A **cluster** is a group of computers that work together.
Spark cuts the data into pieces called **partitions**, and the executors process the pieces in parallel.
The original notebook showed a diagram of the parts here. The kitchen analogy plays the same role:
- the **driver** program coordinates the work, like the head chef;
- the **cluster manager** hands out machines, like a staffing agency;
- the **executors** do the computing, like the line cooks.

Spark offers several APIs. We use the **DataFrame API**, which works with tables.
A **DataFrame** is a table with named columns, spread over the cluster.

## The SparkSession: your waiter

A **SparkSession** is the object that sends your work to Spark. In this notebook it is called `spark`.
The original notebook created it with `SparkSession.builder.master("local[*]").getOrCreate()`.
`master("local[*]")` meant: run Spark on this one laptop and use all its processor cores.

On Databricks you do not choose machines, so we leave out `.master(...)`.
`getOrCreate()` means: give me the session that already exists, or create one if there is none.
Databricks has already created `spark`, so the next cell hands back that same session.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

What each line does (original cells 4 and 5):
- Line 1 borrows the `SparkSession` tool from the `pyspark` library. Session 3 explains `import` in detail.
- Line 3 asks for the existing session and stores it in the variable `spark`.

`type(x)` tells you what kind of object `x` is. Run it on `spark`.

In [ ]:
type(spark)

The word `connect` in the answer means this is a Spark Connect session.
Your notebook is in one place, and the kitchen is in another.
Databricks also set up the machines, Java and Spark for you, and it keeps them running.

## Broken on purpose: run order

The next cell asks Spark to show the taxi trips stored in the variable `trips`.
No cell has created `trips` yet. Predict what will happen, then run it.

In [ ]:
# EXPECT-ERROR: NameError
# Broken on purpose: this cell runs before the cell that creates trips.
trips.show()

**What you should see** (real output, trimmed to the lines that matter):

```text
Traceback (most recent call last):
  File "<cell 4>", line 3, in <module>
    trips.show()
    ^^^^^
NameError: name 'trips' is not defined
```

How to read it, last line first:
- The last line is the most important one. `NameError` is the kind of error.
  The message says Python has never heard of `trips` in this session.
- The `File "<cell ...>"` line points at your cell and repeats the line that failed.
  On Databricks this line looks a little different. Session 4 explains how.
- You get the same error after a restart or a long break.
  When the notebook is disconnected from compute (for example after a long break), Python forgets every variable.

**Fix it.** Run the cell that creates `trips` first. It is the next code cell.

## Reading CSV files

We work with New York City green taxi trips from October to December 2017.
The data comes from the NYC Taxi and Limousine Commission (TLC).
It sits in three CSV files, one per month. CSV stands for comma-separated values.
In a CSV file each line is one record, here one trip, and commas separate the values.

In [ ]:
trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
)

What each line does (original cell 10):
- `spark.read.csv(...)` asks Spark to read CSV files. It returns a DataFrame.
- `f"{BASE}/..."` is an **f-string**. Python replaces `{BASE}` with the value of `BASE`.
- `green_tripdata_2017-1*.csv` is a **glob pattern**. The `*` matches any text.
  So all three monthly files (`-10`, `-11`, `-12`) become one big table.
- `sep=","` says that commas separate the values.
- `header=True` says that the first line holds column names, not data.
  Spark's default is "no header".
- `trips = ...` stores the DataFrame in the variable `trips`.

The cell printed nothing, and Spark has read nothing yet.
It reads the header line when a later cell first needs the column names.

## Displaying a DataFrame with `show`

Now the cell that failed above works. `show()` prints rows as a text table.

In [ ]:
trips.show()

`show()` is an **action**: a command that makes Spark do the work and return a result.
Reading the files did not do the work. `show()` did. Session 4 explains why Spark waits.

How to read the table:
- By default `show()` prints 20 rows and says `only showing top 20 rows`.
- `NULL` means the value is missing. The column `ehail_fee` is empty in every trip.

You can pass the number of rows to show. `vertical=True` prints each row as a list, one column per line.
`truncate=False` prints long values in full instead of cutting them at 20 characters.

In [ ]:
trips.show(3, truncate=False, vertical=True)

**Rows have no fixed order.** A DataFrame is spread over many partitions.
`show()` prints the first rows of whichever partition comes first, and that can change between runs.
To get a fixed order, sort the rows with `orderBy` (Session 3). Sorting a big table takes time.
(The original notebook added a row-number column here with `monotonically_increasing_id`. We skip it.)

## DataFrame columns

`trips.columns` gives the column names as a Python **list**: values in square brackets, in order.

In [ ]:
trips.columns

Indexing a DataFrame with a column name does **not** return that column's values.
It returns a **Column object**: a description of a column, used to build commands.
In the kitchen, a Column object is a dish's name on an order slip, not the food.
You also cannot pick a row by its position.

In [ ]:
trips["VendorID"]

In [ ]:
trips.VendorID

Both spellings (original cells 26 and 27) give the same `Column<'VendorID'>`.

## Counting rows

`count()` returns the number of rows. It is an action, so Spark reads all three files.
To count the columns, use `len(trips.columns)`: `len` counts the items of a list.

In [ ]:
trips.count()

The volume holds 2,705,926 trips. The original course data had 2,706,150 rows.
TLC has since republished its files with 224 fewer trips. So a few numbers here differ from the original.

## Discovery: what kind of thing is it?

`type(x)` tells you what kind of object `x` is. `help(x)` prints the manual of `x`.
Run the next cell. Which result is a DataFrame, which is a list, which is a Column, and which is a whole number?

In [ ]:
print(type(trips))
print(type(trips.columns))
print(type(trips["VendorID"]))
print(type(trips.count()))

Now read the manual of `show`. A **parameter** is a named setting inside the parentheses, like `header=True` in the read.
`True` means yes and `False` means no.
The signature, the line under "Help on method show ...", lists every parameter with its default.
For example, in `vertical: bool = False`, `vertical` takes `True` or `False` (`bool`), and its default is `False`.

**Your question.** What does the parameter `truncate` do when you give it a number?

In [ ]:
help(trips.show)

**Your turn.** Use what you found to show three rows with long values cut to 5 characters.

In [ ]:
# Your code here: show three rows of trips with long values cut to 5 characters

## Session 1 check

Answer in your own words. Then check yourself in the DF Basics module:
Math & Code, the "Your turn" questions under "Getting the SparkSession" and "show, columns and count".

1. You open the notebook after lunch and run only the cell `trips.count()`. What happens, and why?
2. What does `trips["VendorID"]` give you: the vendor IDs, or something else?
3. Which of these make Spark do the work: `spark.read.csv(...)`, `trips.show()`, `trips.columns`, `trips.count()`?

# Session 2: What's inside? Schemas, types and inspection

**Goal.** Read a table's schema, choose the column types, and spot values that went missing.

**The analogy: a printed form.**
A **schema** is like a printed form that every row is copied onto.
Each box on the form says what may go in it, for example "number only".
If a row has "Y" in a number-only box, the clerk does not complain.
The clerk quietly leaves the box blank. We call this a **silent null**.
(A **null** is a missing value. `show()` prints it as `NULL`.)

**Before you run**
- The next code cell loads everything this session needs. Run it first, even after Session 1.

In [ ]:
BASE = "/Volumes/workspace/default/bdcc"

trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
)

## Column types

Every column has a **data type**. The common ones here are:
- `string`: text;
- `integer` (or `int`): a whole number;
- `double` or `float`: a number with a decimal point;
- `timestamp`: a date and a time.

When Spark reads a CSV file, every column is a `string` by default.
`printSchema()` prints the schema: each column's name, its type, and whether it may hold nulls.

In [ ]:
trips.printSchema()

How to read it: `root` is the table itself. Each `|--` line is one column.
`nullable = true` means the column may contain nulls.
Every column is a string, even `fare_amount`. Spark has not looked at the values.

`dtypes` gives the same information as a Python list of (name, type) pairs.

In [ ]:
trips.dtypes

The possible types are listed in [`pyspark.sql.types`](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/data_types.html).

## Giving Spark a schema

You can tell Spark the types yourself. Then Spark does not have to read the data to work out the types.
A **DDL schema string** lists each column and its type, separated by commas.
DDL stands for Data Definition Language, the way databases write down tables and their column types.
Spark matches the pairs to the file's columns by position, not by name, so list every column in the file's order.
(The original notebook first showed a longer way, with `StructType`. We skip it.)

The next cell comes from the original lab notebook. It contains one wrong type, left in on purpose.
Compare each type with the values you saw in `show()` in Session 1. Can you find it?

In [ ]:
schema = """
VendorID INT,
lpep_pickup_datetime TIMESTAMP,
lpep_dropoff_datetime TIMESTAMP,
store_and_fwd_flag INT,
RatecodeID INT,
PULocationID INT,
DOLocationID INT,
passenger_count INT,
trip_distance FLOAT,
fare_amount FLOAT,
extra FLOAT,
mta_tax FLOAT,
tip_amount FLOAT,
tolls_amount FLOAT,
ehail_fee FLOAT,
improvement_surcharge FLOAT,
total_amount FLOAT,
payment_type STRING,
trip_type STRING
"""
trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    schema=schema,
)

What the cell does:
- `"""..."""` is a string that spans several lines. It holds one `name TYPE` pair per line.
- `INT` is a whole number, `FLOAT` a decimal number, `TIMESTAMP` a date and time, `STRING` text.
- `schema=schema` hands the string to the reader. Spark now trusts these types and does not read the data to work them out.

`printSchema()` shows the types you declared. It does not check them against the data.

In [ ]:
trips.printSchema()

Now look at the data itself.

In [ ]:
trips.show(5)

Look at `store_and_fwd_flag`. The file holds the letters `N` and `Y` there.
The schema says `INT`, and a letter is not a whole number.
So Spark wrote NULL in every row, with no error and no warning. That is a silent null.

`describe()` prints a short summary of a column. Its `count` line counts the values that are **not** null.

In [ ]:
trips.describe("store_and_fwd_flag").show()

A count of 0: every flag is gone, and nothing told us.

## Broken on purpose: counting the "Y" trips

A trip has the flag `Y` when the taxi stored it and sent it later. That happens when the meter loses its signal.
The next cell counts the `Y` trips.
`filter(condition)` keeps only the rows where the condition is true. Session 3 covers it fully.
`==` asks "is equal to?". It has two equals signs, because one `=` stores a value.
Predict: what will the count be with this schema?

In [ ]:
# EXPECT-ERROR: NumberFormatException
# Broken on purpose: the schema above declared store_and_fwd_flag as INT.
trips.filter(trips.store_and_fwd_flag == "Y").count()

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 22>", line 3, in <module>
    trips.filter(trips.store_and_fwd_flag == "Y").count()
  ...
pyspark.errors.exceptions.connect.NumberFormatException: [CAST_INVALID_INPUT] The value 'Y' of the type "STRING" cannot be cast to "BIGINT" because it is malformed. Correct the value as per the syntax, or change its target type. Use `try_cast` to tolerate malformed input and return NULL instead. SQLSTATE: 22018
```

How to read it:
- The column is INT, so Spark tries to turn the text `"Y"` into a number to compare them.
  `BIGINT` is Spark's name for a big whole number.
- Spark 4 runs in a strict mode by default, called ANSI mode (after the American National Standards Institute),
  and so did our test run. In strict mode a failed conversion stops the command with `[CAST_INVALID_INPUT]`.
- If your cell prints 0 instead of an error, strict mode is off in your workspace. That 0 is equally wrong, and nothing warns you.
- The error points at this cell, but the real mistake sits in the schema cell above.

**Fix it.** Declare the flag as `STRING`. The next cell is the schema cell above with that one line changed.

In [ ]:
schema_fixed = """
VendorID INT,
lpep_pickup_datetime TIMESTAMP,
lpep_dropoff_datetime TIMESTAMP,
store_and_fwd_flag STRING,
RatecodeID INT,
PULocationID INT,
DOLocationID INT,
passenger_count INT,
trip_distance FLOAT,
fare_amount FLOAT,
extra FLOAT,
mta_tax FLOAT,
tip_amount FLOAT,
tolls_amount FLOAT,
ehail_fee FLOAT,
improvement_surcharge FLOAT,
total_amount FLOAT,
payment_type STRING,
trip_type STRING
"""
trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    schema=schema_fixed,
)
trips.filter(trips.store_and_fwd_flag == "Y").count()

Now the count is 3,650 trips. We kept the broken schema in the variable `schema` for the discovery task below.

**A note on data quality.** The same schema declares `payment_type` and `trip_type` as `STRING`.
Both columns hold whole numbers, so `INT` would fit them better.
This causes no nulls, because any text fits a STRING box. But math on them would need a conversion first.

## Letting Spark guess the types

`inferSchema=True` asks Spark to guess each column's type.
Spark reads all the data one extra time. For each column it tries the strictest type first:
whole number, then decimal number, then date and time. If none fits every value, the column stays text.
The read cell itself returns at once. The extra pass runs at the first command that needs the types,
here `printSchema()`. On big files that pass takes time. A schema that you write yourself avoids it.

In [ ]:
trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    inferSchema=True,
)

In [ ]:
trips.printSchema()

Compare the guesses with our DDL schema:
- IDs and counts became `integer`, and the two dates became `timestamp`.
- `store_and_fwd_flag` became `string`, which is right for `N` and `Y`.
- Amounts became `double`, a more precise decimal type than `float`.
- `ehail_fee` became `string`. The column is empty, so Spark had no number to go on.
- `payment_type` and `trip_type` became `integer`.

## Spark DataFrames and pandas

**pandas** is a Python library for tables that fit on one computer.
`toPandas()` turns a Spark DataFrame into a pandas DataFrame. It is an action.

Be careful with it. It copies **every** row out of Spark into this notebook's own memory.
(With Spark Connect your notebook is not the driver: it is the customer's table in the kitchen picture.)
2.7 million rows can fill that memory and crash the notebook.
You cannot pick rows by position in a Spark table, so first keep a few rows with `limit(10)`, then convert.

In [ ]:
trips.limit(10).toPandas()

On Databricks the pandas table appears as a formatted table, not as plain text.

## Summary statistics: `summary` and `describe`

`summary()` computes statistics for every column:
- `count`: how many values are not null;
- `mean`: the average;
- `stddev`: the standard deviation, a measure of how spread out the values are;
- `min` and `max`: the smallest and largest value;
- `25%`, `50%`, `75%`: the values below which a quarter, half and three quarters of the data fall.

`describe()` gives the same statistics without the percentages. Both read all the data, so they take a while.

In [ ]:
trips.summary().show()

In [ ]:
trips.describe().toPandas()

In [ ]:
trips.summary().toPandas()

The two timestamp columns are missing: `summary()` and `describe()` skip dates.
Look at `fare_amount` instead. The smallest fare is negative and the largest is over $6,000.
Odd values like these are common in real data. Always check your data before you trust it.

## Aha: summaries of text columns

What if the columns were text? We read the files again with no schema, so every column is a string.

In [ ]:
trips_text = spark.read.csv(f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv", header=True)
trips_text.describe("fare_amount", "PULocationID", "lpep_pickup_datetime").show()

`mean` and `stddev` still come out as numbers for the fare and the zone: Spark converts the text for those two.
But `min` and `max` compare text, character by character, like words in a dictionary:
- For `PULocationID` the "largest" value is `99`, although the real largest zone number is 265.
  `"9"` comes after `"2"`, so `"99"` beats `"265"`.
- For `fare_amount` the text max is `99.5` and the text min is `-0.01`.
  The real values, in the summary above, are 6003.5 and -400.09.

This is why the column types matter.

The dates are a lucky exception. Text like `2017-10-01 00:27:41` sorts in time order.
So `min` and `max` show the real range: some pickups are dated 2008 or 2018, outside October to December 2017.
Those are bad clock values in the taxi meters.

## Discovery: make Spark fail loudly

A silent null is worse than an error, because nothing tells you to look.
Run `help(spark.read.csv)` and look for a parameter about rows that do not fit the schema.
Tip: the signature, the line under "Help on method csv ...", lists every parameter.

In [ ]:
help(spark.read.csv)

The manual lists `mode` but does not explain its values.
Near the middle it points to the "Data Source Option" page instead:
https://spark.apache.org/docs/latest/sql-data-sources-csv.html#data-source-option

**Your question.** Open that page and find `mode`. Which value makes Spark stop at the first row that does not fit the schema?
Then run the next cell, which uses that value with the broken INT schema.

In [ ]:
# EXPECT-ERROR: SparkException
# The broken INT schema again, read with the mode you found.
trips_strict = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    schema=schema,
    mode="FAILFAST",
)
trips_strict.show(5)

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 32>", line 10, in <module>
    trips_strict.show(5)
  ...
pyspark.errors.exceptions.connect.SparkException: [FAILED_READ_FILE.NO_HINT] Encountered error while reading file dbfs:/Volumes/workspace/default/bdcc/nyctaxi/green_tripdata_2017-10.csv.  SQLSTATE: KD001
```

(The file path is as our local run printed it; Databricks may print it a little differently.)

The last line only says that Spark could not read one of the files.
The reason is further down the full error, in the `Caused by:` lines. In our run they read:

```text
Caused by: org.apache.spark.SparkException: [MALFORMED_RECORD_IN_PARSING.WITHOUT_SUGGESTION] Malformed records are detected in record parsing: [2,1506817661000000,1506817965000000,null,1,264,193,1,0.0,2.5,0.5,0.5,0.0,0.0,null,0.3,3.8,2,1].
Parse Mode: FAILFAST. To process malformed records as null result, try setting the option 'mode' as 'PERMISSIVE'.  SQLSTATE: 22023
Caused by: java.lang.NumberFormatException: For input string: "N"
```

The record in square brackets is the first trip, with the bad value shown as `null` in fourth place.
`For input string: "N"` names the culprit: an `N` that should have been a number.
Loud is better than silent. The fix is the same as before: declare the flag as `STRING`.

## Session 2 check

Answer in your own words. Then check yourself in the DF Basics module:
Math & Code, the "Your turn" questions under "DDL schema strings and silent nulls" and "limit, then toPandas".

1. After a plain `spark.read.csv(path, header=True)`, what type does every column have? Name two ways to get numbers instead.
2. A schema declares `store_and_fwd_flag INT`, but the file holds `N` and `Y`. What does `show()` print in that column, and why is that worse than an error?
3. Why write `trips.limit(10).toPandas()` instead of `trips.toPandas()`?

# Session 3: Imports and method chains

**Goal.** Borrow Spark's tools safely, and build an answer step by step with a method chain.

**The analogies.**
- **Importing** is borrowing a tool from a shared toolbox.
  Python has its own tools called `sum`, `max` and `round`. PySpark's toolbox has tools with the same names.
  Writing `F.sum` puts the toolbox's name on the tool, so you never grab Python's own `sum` by mistake.
- A **method chain** is an assembly line.
  Each station takes a table, changes it, and hands a **new** table to the next station.

**Before you run**
- The next code cell loads the trips with `inferSchema=True`, so the columns hold numbers.
  The read itself returns at once. The first cell that needs the column types makes Spark read
  all the data once more to guess them, so that cell takes a little longer.

In [ ]:
BASE = "/Volumes/workspace/default/bdcc"

trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    inferSchema=True,
)

## Imports: borrowing tools

A **function** is a named piece of code that you call with parentheses, like `round(4.567, 1)`.
The values inside the parentheses are its **arguments**.
Python has **built-in** functions that need no import at all, such as `round`, `max` and `sum`.
A **module** is a named collection of ready-made tools. An `import` line loads a module's tools into your notebook.

In [ ]:
print(round(4.567, 1))    # Python's own round: one decimal place
print(max(3, 7))          # Python's own max: the larger number
print(sum([1, 2, 3]))     # Python's own sum: adds up a list

There are three ways to borrow PySpark's functions:
- `from pyspark.sql.functions import desc` borrows one tool, `desc`, under its bare name.
- `from pyspark.sql.functions import *` borrows **every** tool under its bare name. This is risky, as you will see.
- `from pyspark.sql import functions as F` borrows the whole toolbox under the short name `F`.
  You then write `F.desc`, `F.col` or `F.sum`. **This is the style we recommend.**

In [ ]:
from pyspark.sql import functions as F

type(F)

`F` is a `module`: the toolbox itself. New code in this notebook uses `F.` from now on.

## Choosing columns with `select`

A **method** is a function that belongs to an object. You call it with a dot, as in `trips.select(...)`.
`select` keeps only the columns you name. It returns a new DataFrame and leaves `trips` unchanged.

In [ ]:
trips.select("PULocationID").show()

In [ ]:
trips.select("PULocationID", "DOLocationID").show()

`PULocationID` is the pickup zone and `DOLocationID` the drop-off zone. NYC is split into 265 numbered taxi zones.

`describe` can also summarize only the columns you name (original cell 58).

In [ ]:
trips.describe("PULocationID", "DOLocationID").show()

## Frequent items

`freqItems` finds values that appear in at least a given share of the rows. That share is called the **support**.
`0.01` means 1%, so the next cells find zones that appear in at least 1% of the trips.
The method is approximate: the list can include a few values below the threshold.

In [ ]:
trips.freqItems(["PULocationID"], 0.01).show()

In [ ]:
trips.freqItems(["PULocationID", "DOLocationID"], 0.01).show()

The result is one row that holds a list (Spark calls it an **array**). The list is cut off at 20 characters.
To see each zone on its own row, original cell 63 uses `explode`.
`explode` turns each element of an array into its own row.

Original cell 63 starts with `from pyspark.sql.functions import explode`.
That line borrows only `explode`, under its bare name. We keep the line as the original had it.
New code writes `F.explode` instead, so you can see which toolbox the tool came from.

In [ ]:
from pyspark.sql.functions import explode

(
    trips.freqItems(["PULocationID"], 0.01)
    .withColumn("freq_items", explode("PULocationID_freqItems"))
    .drop("PULocationID_freqItems")
    .show()
)

**Reading the chain, one station at a time.** The outer parentheses let the chain span several lines.
1. `trips.freqItems(["PULocationID"], 0.01)` makes a one-row table holding the array.
2. `.withColumn("freq_items", explode(...))` adds a column `freq_items`, with one row per array element.
3. `.drop("PULocationID_freqItems")` removes the array column.
4. `.show()` prints the result. It is the only action in the chain.

To check a chain, run it piece by piece. Cut it after a station and add `.show()`.
Here is the chain up to station 2. The array column is still there.

In [ ]:
(
    trips.freqItems(["PULocationID"], 0.01)
    .withColumn("freq_items", explode("PULocationID_freqItems"))
    .show(5)
)

Original cell 64 reaches the same result another way.
It replaces the array column with its exploded version, then renames the column with `withColumnRenamed`.

In [ ]:
from pyspark.sql.functions import explode

(
    trips.freqItems(["PULocationID"], 0.01)
    .withColumn("PULocationID_freqItems", explode("PULocationID_freqItems"))
    .withColumnRenamed("PULocationID_freqItems", "freq_items")
    .show()
)

## Grouping, counting and sorting

How many trips start in each zone? Original cell 66 answers with a chain of three stations:
- `groupby(column)` collects the rows that share a value into groups.
  (`groupBy`, with a capital B, is another spelling of the same method.)
  The groups are not a table yet. They wait for a summary, such as `count()`.
- `count()` counts the rows in each group, in a new column called `count`.
- `orderBy(desc("count"))` sorts by that column. `desc` means descending: largest first.

The cell keeps its original import line. It borrows `explode` and `desc` under their bare names.

In [ ]:
from pyspark.sql.functions import explode, desc

(trips.groupby(trips["PULocationID"]).count().orderBy(desc("count")).show())

**Ties.** If two zones had the same count, their order would be up to Spark. It could change between runs.
A second sort key, called a **tie-break**, decides the order of rows whose first key is equal.
Here is the same chain in the `F` style, with the zone number as the tie-break.

In [ ]:
(
    trips.groupBy("PULocationID")
    .count()
    .orderBy(F.desc("count"), "PULocationID")
    .show(5)
)

Group-by works with other summaries too. Original cell 68 does four things:
1. `withColumn("total_amount", trips.total_amount.cast("float"))` replaces `total_amount` with a copy converted to `float`.
   **Casting** means converting a column to another type.
2. `groupby("VendorID")` makes one group per taxi company. (`VendorID` 1 and 2 are two companies that run the meters.)
3. `sum("total_amount")` adds up the amounts in each group.
4. `describe()` then summarizes those two sums.

In [ ]:
(
    trips.withColumn("total_amount", trips.total_amount.cast("float"))
    .groupby("VendorID")
    .sum("total_amount")
    .describe()
    .show()
)

In [ ]:
(
    trips.withColumn("total_amount", trips.total_amount.cast("float"))
    .groupby("VendorID")
    .count()
    .show()
)

More column functions are listed in [pyspark.sql.functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html).

## Filtering rows

`filter` keeps the rows where a **condition** is true.
`trips.total_amount > 1000` builds a condition: true or false for each row.
It is a Column, not one single yes-or-no answer.

In [ ]:
(trips.filter(trips.total_amount > 1000).count())

Five trips cost more than $1,000. That is worth a second look in a real analysis.

Combine conditions with `&` (and), `|` (or) and `~` (not).
Put each condition in its own parentheses. Without them, Python groups the `&` first and the result is wrong.
`==` asks "is equal to?". It has two equals signs, because one `=` stores a value.
Original cell 74 keeps the trips from zone 264 to zone 193:

In [ ]:
(
    trips.filter(
        (trips["PULocationID"] == 264) & (trips["DOLocationID"] == 193)
    )
    .limit(10)
    .toPandas()
)

`F.col("name")` is one more way to point to a column.
`trips.PULocationID`, `trips["PULocationID"]` and `F.col("PULocationID")` all point to the same column.
Here is an "or": the trips that start in zone 7 or in zone 33.

In [ ]:
trips.filter((F.col("PULocationID") == 7) | (F.col("PULocationID") == 33)).count()

## Wrapping a chain in a function

The graded problems ask you to write your own functions: your assembly line, saved under a name.
- `def name(parameter):` starts it. The indented lines below it are its body.
- In a `def`, a **parameter** such as `df` is a placeholder for the value you pass in.
- `return value` hands the result back to whoever called the function.

In [ ]:
def top_pickups(df):
    """Return the pickup zones with their trip counts, busiest first."""
    return df.groupBy("PULocationID").count().orderBy(F.desc("count"), "PULocationID")


top_pickups(trips).show(5)

### Broken on purpose: the missing `return`

The next function forgot its `return`. Predict the error, then run it.

In [ ]:
# EXPECT-ERROR: AttributeError
# Broken on purpose: the function has no return line.
def top_pickups(df):
    df.groupBy("PULocationID").count().orderBy(F.desc("count"), "PULocationID")


top_pickups(trips).show(5)

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 52>", line 7, in <module>
    top_pickups(trips).show(5)
    ^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'NoneType' object has no attribute 'show'
```

A function without `return` hands back `None`, Python's value for "nothing".
The chain inside still ran, but its result was thrown away.
An **AttributeError** means the thing before the dot has no such attribute.
(An attribute is anything you reach with a dot.) `None` has no `show` method.

**Fix it.** Put `return` in front of the chain, as in the cell before.

In [ ]:
def top_pickups(df):
    return df.groupBy("PULocationID").count().orderBy(F.desc("count"), "PULocationID")


top_pickups(trips).show(5)

## Broken on purpose: the wildcard import

Earlier, `round(4.567, 1)` gave `4.6`. The next cell borrows **every** PySpark function with `*`, then rounds again.
Predict what happens.

In [ ]:
# EXPECT-ERROR: AssertionError
# Broken on purpose: the wildcard import replaces Python's own round.
from pyspark.sql.functions import *

round(4.567, 1)

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 54>", line 5, in <module>
    round(4.567, 1)
  ...
AssertionError
```

A bare `AssertionError` with no message: one of the least helpful errors in Python.
An **AssertionError** means that a check inside a tool failed.
Here is what happened. The `*` import brought in PySpark's own `round`, which replaced Python's `round`.
PySpark's `round` expects a column, not the number `4.567`, so it fails.
The wildcard import replaced 12 of Python's built-ins: `abs`, `ascii`, `bin`, `filter`, `hash`, `hex`, `max`, `min`, `pow`, `round`, `slice` and `sum`.
`max(3, 7)` now fails too, with `TypeError: max() takes 1 positional argument but 2 were given`.
A **TypeError** says a function was handed something it cannot use. PySpark's `max` expects one column.

**Fix it.** Take Python's versions back from the `builtins` module, where they live.
In your own code, avoid `import *`. Use `F.round` when you mean Spark's version.

In [ ]:
from builtins import abs, ascii, bin, filter, hash, hex, max, min, pow, round, slice, sum

print(round(4.567, 1), max(3, 7), sum([1, 2, 3]))

## Broken on purpose: `and` between conditions

In plain Python you write `and`. On Spark columns, `and` does not work. Predict the error, then run the cell.

In [ ]:
# EXPECT-ERROR: PySparkValueError
# Broken on purpose: Python's "and" between two Spark conditions.
trips.filter((F.col("fare_amount") > 10) and (F.col("tip_amount") > 0)).count()

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 56>", line 3, in <module>
    trips.filter((F.col("fare_amount") > 10) and (F.col("tip_amount") > 0)).count()
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  ...
pyspark.errors.exceptions.base.PySparkValueError: [CANNOT_CONVERT_COLUMN_INTO_BOOL] Cannot convert column into bool: please use '&' for 'and', '|' for 'or', '~' for 'not' when building DataFrame boolean expressions.
```

`bool` (short for Boolean) is Python's type for one true-or-false value.
A "boolean expression" is a condition like the ones above.
Python's `and` needs one plain true or false. A Spark condition holds one answer per row, so Python cannot decide.
The message tells you the fix: `&` for `and`, `|` for `or`, `~` for `not`.

**Fix it.**

In [ ]:
trips.filter((F.col("fare_amount") > 10) & (F.col("tip_amount") > 0)).count()

## Discovery: find a function nobody taught you

Task: round each pickup time down to the start of its hour, for example `08:47:12` becomes `08:00:00`.
`dir(F)` lists every name in the toolbox. The next cell keeps only the names that contain "hour".

In [ ]:
[name for name in dir(F) if "hour" in name]

Neither `hour` nor `hours` rounds a time down. Try a wider search for names with "date" or "trunc",
for example `[name for name in dir(F) if "trunc" in name]`.
One candidate is `date_trunc`. Read its manual.

In [ ]:
help(F.date_trunc)

**Your question.** Which two arguments does `F.date_trunc` take, and in which order?
Use it to show each pickup time next to its rounded-down hour.
Name the new column `pickup_hour` with `.alias("pickup_hour")`. `alias` gives a result column a name.
Without it, the header would be the whole expression.

In [ ]:
# Your code here: select lpep_pickup_datetime and its rounded-down hour

## Session 3 check

Answer in your own words. Then check yourself in the DF Debugging module:
Math & Code, the "Your turn" questions under "Three ways to import", "Group, count and sort" and "Filtering rows".

1. After `from pyspark.sql.functions import *`, why does `round(4.567, 1)` fail? How does the `F.` style prevent it?
2. Fix this line: `trips.filter(trips.fare_amount > 10 and trips.tip_amount > 0)`.
3. Two zones have exactly the same trip count. Write an `orderBy` that puts the busiest zone first
   and always lists tied zones in the same order, smallest zone number first.

# Session 4: Debugging, when Spark complains

**Goal.** Read an error message calmly, find the line that matters, and fix the cause.

**The analogy: a shopping list.**
Writing a shopping list does not buy anything. If you misspell "tomatoe" on the list, nothing stops you.
You find out only at the store, when you try to buy it.
Spark works the same way: most commands only add a line to the list. The **action** is the trip to the store.
When something goes wrong, Spark hands you a long receipt, the **traceback**.
The line that matters is near the bottom.

**Before you run**
- The next code cell loads everything this session needs, with `inferSchema=True`.

In [ ]:
BASE = "/Volumes/workspace/default/bdcc"
from pyspark.sql import functions as F

trips = spark.read.csv(
    f"{BASE}/nyctaxi/green_tripdata_2017-1*.csv",
    sep=",",
    header=True,
    inferSchema=True,
)

## Transformations, actions and the plan

Spark commands come in two kinds:
- A **transformation**, such as `select`, `filter`, `orderBy` or `withColumn`, describes a new table.
  It only writes a new step on the list, so it returns at once.
- An **action** asks for a result: `show`, `count`, `toPandas`, `first`.
  Only then does Spark do the work.

Waiting until an action is called **lazy evaluation**. The list of steps is called the **plan**.
`explain()` prints the plan without running it. Original cell 98 does this.
(The original used `desc` from an earlier import. This session starts fresh, so we write `F.desc`.)

In [ ]:
trips2 = trips.groupby(trips["PULocationID"]).count().orderBy(F.desc("count"))
trips2.explain()

Read a plan from the bottom up. The bottom line reads the CSV files.
The lines above it count per zone and then sort.
You do not need to understand every word. The point is that a plan exists before anything runs.
On Databricks the plan looks different, because Databricks uses its own faster engine, called Photon.

Now an action runs the plan:

In [ ]:
trips2.show(5)

## Broken on purpose: a typo that waits

Spark Connect, which Databricks uses, does not check column names in most transformations.
It checks them when an action runs. So a typo in a `select` passes silently, and the error appears later.
The next cell selects a column with a misspelled name. Run it.

In [ ]:
pu = trips.select("PULocatonID")
type(pu)

No error: `pu` is a DataFrame, waiting on the list. Now ask for the data. Predict what happens.

In [ ]:
# EXPECT-ERROR: AnalysisException
# Broken on purpose: pu was built from a misspelled column name.
pu.show()

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 65>", line 3, in <module>
    pu.show()
  ...
pyspark.errors.exceptions.connect.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `PULocatonID` cannot be resolved. Did you mean one of the following? [`PULocationID`, `DOLocationID`, `VendorID`, `tip_amount`, `RatecodeID`]. SQLSTATE: 42703;
'Project ['PULocatonID]
```

**How to read a traceback.** A **traceback** is Python's error report: the chain of calls from your cell
down to the failure. It is the long receipt of the shopping-list picture. Read two parts, last line first:
1. **The last line** says what went wrong: first the error's name, then its message.
   - `AnalysisException` means Spark could not make sense of the plan.
   - `[UNRESOLVED_COLUMN.WITH_SUGGESTION]` is Spark's more exact error code, called the **error class**.
     Search the web for it if you are stuck.
   - The message names the bad column, `PULocatonID`, and offers a **"Did you mean"** list.
     The first suggestion is usually the right one.
   - Under the message, Spark prints its plan, last step on top. `'Project ['PULocatonID]` is the step
     with the bad name. `Project` is Spark's word for `select`.
2. **Your cell's frame** says where the problem surfaced. Each `File ..., line N` entry is a **frame**:
   one step of the chain of calls. Yours is the `File "<cell ...>"` line, with the code under it.
   It points at `pu.show()`, the action, not at the `select` line with the typo.

Everything in between, the `...` here, is Spark's own machinery. You can skip it.
On Databricks the full error also has a `JVM stacktrace:` block. JVM stands for Java Virtual Machine,
the program that Spark runs on. Skip that block too.

## Databricks shows errors slightly differently

The errors in this notebook were produced by Spark 4.0.1 running outside Databricks.
Databricks prints the same messages with slightly different decoration:
- your cell's frame appears as `File <command-…>, line N` instead of `File "<cell N>", line N`;
- the error's name appears as `AnalysisException:` without the long `pyspark.errors.exceptions.connect.` in front.
The error class in square brackets and the message are the same.

**Fix it.** Correct the spelling and run the action again.

In [ ]:
pu = trips.select("PULocationID")
pu.show(5)

**Not every command waits.** A few commands may ask Spark for the table's column list straight away:
`trips["name"]`, `trips.name`, and `groupBy("name")` with the name in quotes.
If the table already holds a typo, the error then appears at that line, before any action.
Either way, the mistake sits in the failing line or in a line above it.

**A surprise: capital letters do not matter.** Spark matches column names without caring about upper or lower case.
So `"PULocationId"` (small `d`) works. The header shows the spelling you typed.
A missing or extra letter, as in the broken cell, is a real typo.

In [ ]:
trips.select("PULocationId").show(3)

## The four errors you will meet most

| What you see | What it usually means | First thing to check |
|---|---|---|
| `AnalysisException` with `UNRESOLVED_COLUMN` | A column name is misspelled, or the column is nested (Session 6) | `trips.columns` or `printSchema()` |
| `NameError: name '...' is not defined` | A cell was skipped, Python restarted, or an import is missing | Run the cells above in order |
| `TypeError` | A value of the wrong kind was passed to a function | `type(...)` of each argument |
| No error, but nulls or a wrong number | A **silent null**: the schema does not fit the data (Session 2) | `printSchema()`, `show(5)`, `describe()` |

### Broken on purpose: a `TypeError`

The next cell tries to keep trips that cost more than $1,000. Predict the error.

In [ ]:
# EXPECT-ERROR: TypeError
# Broken on purpose: the column name is in quotes, so Python sees plain text.
trips.filter("total_amount" > 1000).count()

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 68>", line 3, in <module>
    trips.filter("total_amount" > 1000).count()
                 ^^^^^^^^^^^^^^^^^^^^^
TypeError: '>' not supported between instances of 'str' and 'int'
```

This error comes from Python, before Spark sees anything.
`"total_amount"` is a piece of text (a `str`), and Python cannot compare text with the number 1000.
No `...` line appears, because the error happened right in your cell.

**Fix it.** Compare the **column** with the number.

In [ ]:
trips.filter(trips.total_amount > 1000).count()

## Your inspection toolkit

When a result looks wrong, look at the data before you change any code:
- `trips.printSchema()`: each column's name and type, as a tree;
- `trips.dtypes`: the same, as a list of (name, type) pairs;
- `trips.show(5)`: a few real rows;
- `trips.columns`: the exact spelling of every name;
- `trips.count()`: how many rows there are;
- `trips.limit(10).toPandas()`: a small sample as a pandas table.

In [ ]:
trips.dtypes[:6]

`[:6]` keeps the first six items of a Python list.

## Asking the Databricks Assistant

Databricks has a built-in artificial intelligence (AI) helper, the **Assistant**. It answers questions about your notebook.
Its answers are only as good as your question. A good debugging question contains:
1. the code you ran;
2. the **full** error, copied as text;
3. the output of `printSchema()` for the DataFrame involved;
4. what you expected to happen.

Treat its answer like advice from a classmate: test it, and make sure you understand it.

## Discovery: other ways to filter

Read the manual of `filter`. It uses the word SQL: Structured Query Language,
the classic language for asking questions of tables. Find two things:
1. Besides a Column condition, what else does `filter` accept?
2. Which other method does exactly the same as `filter`?

In [ ]:
help(trips.filter)

**Your turn.** Count the trips with `total_amount > 1000` in the other way the manual shows. Then count them with the twin method.

**Then write.** Draft a question for the Assistant about the `pu.show()` error above. Use the four parts.

In [ ]:
# Your code here: count the trips over 1000 in the other way, then with the twin method

## Session 4 check

Answer in your own words. Then check yourself in the DF Debugging module:
Math & Code, the "Your turn" questions under "The plan and the action" and "Four debugging steps".

1. `pu = trips.select("PULocatonID")` runs with no error, although the name is misspelled. Why? Where will the error appear?
2. The full traceback of the failed `pu.show()` is very long. Which two parts do you read first, and in which order?
3. `trips.filter("total_amount" > 1000)` raises `TypeError: '>' not supported between instances of 'str' and 'int'`.
   What is wrong? Write two correct versions: one that compares the column, and one that uses a SQL text.

# Session 5: Parquet, files and the first graded problems

**Goal.** Read and write Parquet files, compare them with CSV, and solve your first graded problems.

**The analogy: a novel and a filing cabinet.**
A CSV file is like a novel. To find every character's age, you read every page from start to end.
Parquet is like a filing cabinet with one drawer per column.
To average the fares, you open only the fare drawer.

**Row and column layouts.** CSV stores data row by row, so each trip's values sit together: a **row layout**.
That is handy when you need whole records. But analysis usually needs a few columns over all rows.
**Apache Parquet** stores the data column by column: a **columnar layout**. Each column is compressed (packed smaller),
and the file remembers each column's type. Spark, pandas and many other tools read Parquet.

**Before you run**
- This session starts by **restarting Python on purpose**, as if you came back tomorrow.
  Everything defined above is forgotten. The second code cell loads what this session needs.

In [ ]:
# Restart Python: every variable and import from earlier sessions is forgotten.
# (On Databricks you can also use the compute menu: "Restart Python" or "Clear state".)
dbutils.library.restartPython()

In [ ]:
BASE = "/Volumes/workspace/default/bdcc"
from pyspark.sql import functions as F

trips = spark.read.parquet(f"{BASE}/nyctaxi/green_2017")

What each line does:
- Lines 1 and 2 set `BASE` and the `F` toolbox again, because the restart erased them.
  `F` is the short name for Spark's toolbox of column tools, so `F.desc` means "`desc` from that toolbox".
- `spark.read.parquet(...)` reads Parquet (original cell 79). No `header` or schema is needed:
  the names and types are stored in the files.

## A folder, not a file

`nyctaxi/green_2017` is a **folder** of Parquet files, not a single file. Spark reads the whole folder as one table.
`dbutils` is Databricks' toolbox for files, and `dbutils.fs.ls(path)` lists a folder.
(The original notebook used the terminal command `!ls -lr` here.)
Each entry has a `name` and a `size` in bytes. The `for` line repeats `print` once for each entry.

In [ ]:
for info in dbutils.fs.ls(f"{BASE}/nyctaxi/green_2017"):
    print(info.name, info.size)

`_SUCCESS` is an empty marker file. Spark writes it last, when the whole folder has been saved.
The `part-...` files hold the data. Your file names and sizes will differ: Spark picks them when it writes.

Parquet keeps the types, so `printSchema()` shows numbers and timestamps straight away.

In [ ]:
trips.printSchema()

In [ ]:
trips.count()

The same 2,705,926 trips as the CSV files.

## Writing Parquet, split into partition folders

`write.parquet(path)` saves a DataFrame as a Parquet folder.
- `mode="overwrite"` replaces the folder if it already exists.
- `partitionBy=[...]` splits the rows into sub-folders named `column=value`, one per value.
  Each one is a **partition folder**: it holds only the rows with that value.
  Queries that filter on those columns can skip whole folders.

Original cell 84 saves 100 trips, split by pickup zone and then by drop-off zone.
We write into the `output` folder of our volume. The first write creates that folder.

In [ ]:
(
    trips.limit(100).write.parquet(
        f"{BASE}/output/sample_write",
        mode="overwrite",
        partitionBy=["PULocationID", "DOLocationID"],
    )
)

In [ ]:
folders = dbutils.fs.ls(f"{BASE}/output/sample_write")
print(len(folders), "entries, for example:")
for info in folders[:3]:
    print(" ", info.name)
for info in dbutils.fs.ls(folders[0].path):
    print("    inside", folders[0].name, "->", info.name)

Each pickup zone got a folder like `PULocationID=7/`, with drop-off folders inside it.
The values now live in the folder names. When Spark reads the folder back, it rebuilds the two columns from the names
and puts them at the **end** of the schema:

In [ ]:
spark.read.parquet(f"{BASE}/output/sample_write").printSchema()

## Problem 1 (discussion): CSV versus Parquet

The cells in this section re-run cells from Sessions 2 and 3, but on Parquet.

**Problem 1.** Compare the results and execution times of the cells in this section with the matching cells of
Sessions 2 and 3, which read CSV. Which operations give the same or different results?
Which are about as fast, and which are much faster or slower?
Can you say which kinds of operations benefit from Parquet files?
(Databricks shows each cell's run time under the cell.)

**Start of section for Problem 1**

In [ ]:
trips.columns

In [ ]:
trips.describe().toPandas()

In [ ]:
trips.describe("PULocationID").show()

### Broken on purpose: running a section on its own

The next cell is original cell 91, unchanged. In the original notebook it worked,
because an earlier section had imported `explode`. Python was restarted at the start of this session.
Predict what happens.

In [ ]:
# EXPECT-ERROR: NameError
(
    trips.freqItems(["PULocationID"], 0.01)
    .withColumn("freq_items", explode("PULocationID_freqItems"))
    .drop("PULocationID_freqItems")
    .show()
)

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 84>", line 4, in <module>
    .withColumn("freq_items", explode("PULocationID_freqItems"))
                              ^^^^^^^
NameError: name 'explode' is not defined
```

The cell relied on an import from a cell that has not run since the restart.
The next original cell, 92, has the same problem with `desc`.
A section that works only after other sections have run is fragile.

**Fix it.** Import what the section needs at the top of the section. Two ways work:
- add `from pyspark.sql.functions import explode, desc` (the next cell does this); or
- write `F.explode` and `F.desc`, which need only the `F` import in this session's setup cell.

In [ ]:
from pyspark.sql.functions import explode, desc

(
    trips.freqItems(["PULocationID"], 0.01)
    .withColumn("freq_items", explode("PULocationID_freqItems"))
    .drop("PULocationID_freqItems")
    .show()
)

In [ ]:
(trips.groupby(trips["PULocationID"]).count().orderBy(desc("count")).show())

In [ ]:
(
    trips.withColumn("total_amount", trips.total_amount.cast("float"))
    .groupby("VendorID")
    .sum("total_amount")
    .describe()
    .show()
)

In [ ]:
(
    trips.filter(
        (trips["PULocationID"] == 264) & (trips["DOLocationID"] == 193)
    )
    .limit(10)
    .toPandas()
)

**End of section for Problem 1**

**Your observations.** Write your answer to Problem 1 here: same or different results, faster or slower, and why.

## The graded-problem workflow

Each graded problem has three cells:
1. a **spec**: a text cell that says exactly what your function must return;
2. a **stub**: a code cell with the function's first line and `raise NotImplementedError()`.
   Replace the `raise` line with your code;
3. a **test**: a code cell that calls your function and checks the result.

The tests use `assert_equal` from the `numpy` library. It says nothing when the values match.
When they differ, it raises an `AssertionError` that shows both values.

In [ ]:
from numpy.testing import assert_equal, assert_array_almost_equal

assert_equal(trips.count(), 2705926)
print("passed")

### What a failing test looks like

The original course data had 2,706,150 trips. Here is a test that still expects that number.

In [ ]:
# EXPECT-ERROR: AssertionError
# Broken on purpose: the expected value is from the original data.
assert_equal(trips.count(), 2706150)

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 90>", line 3, in <module>
    assert_equal(trips.count(), 2706150)
  ...
AssertionError: 
Items are not equal:
 ACTUAL: 2705926
 DESIRED: 2706150
```

`ACTUAL` is what your code produced. `DESIRED` is what the test expected.
When a problem test fails, compare the two, then check your code against the spec.

## Tools for Problems 2 and 3

**`agg` with a named result.** After `groupBy`, `agg(...)` computes one summary value per group:
an **aggregation**. `F.mean("column")` is the average. `.alias("new name")` names the result column.

In [ ]:
(
    trips.groupBy("VendorID")
    .agg(F.mean("tip_amount").alias("mean tip"))
    .orderBy("VendorID")
    .show()
)

**`first()` and the Row object.** `first()` is an action. It returns the first row as a **Row**, a small record.
You can read its values by name or by position. Positions count from 0, so `row[0]` is the first value.
A **tuple** is a fixed list of values in round brackets, such as `(2, 2218153)`. `tuple(row)` turns the Row into a tuple.

In [ ]:
row = trips.groupBy("VendorID").count().orderBy(F.desc("count")).first()
row

In [ ]:
print(row["count"], row[0], tuple(row))

## Discovery: what can a group do?

What does `trips.groupby(...)` return? Is it a DataFrame? And which methods does it have?
`type(x)` tells you what kind of thing `x` is. `dir(x)` lists the names inside `x`, including its methods.
The last line of the next cell keeps only the names that do not start with `_`, Python's hidden helpers.

In [ ]:
grouped = trips.groupby("PULocationID")
print(type(grouped))
[name for name in dir(grouped) if not name.startswith("_")]

**Your question.** Find `avg` and `mean` in the list. In the `F` toolbox, are `F.mean` and `F.avg` the same thing?
The next cell asks three things at once, separated by commas.
`a is b` asks whether two names point to the very same thing.
Run it and explain what you see. Will the two give the same average?

In [ ]:
F.mean is F.avg, F.mean("fare_amount"), F.avg("fare_amount")

## Problem 2 (graded)

Create a function `pudo` that accepts `trips` and returns a triple: the `PULocationID`, the `DOLocationID`
and the count of the most frequent `PULocationID`-`DOLocationID` pair.

**The single-chain rule is relaxed: you may use several statements.**
(The original asked for one continuous method chain.)

In [ ]:
def pudo(trips):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
from numpy.testing import assert_equal, assert_array_almost_equal

pudo_max = pudo(trips)
assert_equal(len(pudo_max), 3)
assert_equal(pudo_max[0], 7)
assert_equal(pudo_max[1], 7)
assert_equal(pudo_max[2], 38832)
print("Problem 2 test passed")

## Problem 3 (graded)

Create a function `ave_amount` that accepts `trips` and returns a Spark DataFrame with columns `PULocationID`
and `mean fare amount`, which contains the average fare amount per `PULocationID`. Sort by `PULocationID`.

**The single-chain rule is relaxed: you may use several statements.**

**Data note.** The original test expected 255 pickup zones. The TLC files now have 224 fewer trips,
and 4 zones no longer appear, so the test expects 251. The first ten averages did not change.

In [ ]:
def ave_amount(df_trips):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
mean_fare = ave_amount(trips).toPandas()

assert_equal(len(mean_fare), 251)
assert_equal(mean_fare.columns.tolist(), ["PULocationID", "mean fare amount"])
assert_equal(
    mean_fare["PULocationID"].to_numpy()[:10], [1, 2, 3, 5, 6, 7, 8, 9, 10, 11]
)
assert_array_almost_equal(
    mean_fare["mean fare amount"].to_numpy()[:10],
    [
        72.33266667,
        61.5,
        17.55557656,
        42.83333333,
        35.77777778,
        10.08359857,
        13.625,
        24.45208845,
        19.82410309,
        21.73911883,
    ],
)
print("Problem 3 test passed")

## Session 5 check

Answer in your own words. Then check yourself in the DF Formats module:
Math & Code, the "Your turn" questions under "Reading, listing and writing Parquet" and "Problem 1: CSV versus Parquet".

1. Why is averaging `fare_amount` faster on Parquet than on CSV files that hold the same trips?
2. After a restart you run only the Problem 1 cell that uses `explode(...)`. What happens? Give two fixes.
3. The notebook saves 100 trips with `partitionBy=["PULocationID", "DOLocationID"]`. What does the saved folder look like?

# Session 6: Nested JSON, text files and the capstone

**Goal.** Read nested JSON data, reach inside it with dot paths, and find out which file each row came from.

**The analogy: folders inside folders.**
In a JSON record, a value can hold more named values, like a folder holding files and more folders.
This folder is only a picture: it lives inside each row, not on disk like Session 5's Parquet folders.
`"actor.login"` is a path to a value, like `Documents/School/notes` on your computer.

**The data: GH Archive.** GH Archive records public activity on GitHub, the code-sharing website.
Each **event** is one thing a user did. Examples are starring a repository (marking it as a favourite)
and pushing code (uploading saved changes).
(A **repository**, or repo, is a project's folder of code on GitHub.)
We use three hours of events from 1 January 2015: 35,061 events in three files.
(The original notebook used a Twitter sample stored on the course server. It is not public, so we use GH Archive.)

**JSON** (JavaScript Object Notation) writes records as `{"name": value, ...}`.
**JSON Lines** puts one JSON record on each line. Our files are JSON Lines compressed with **gzip**,
hence the ending `.json.gz`. Spark unpacks gzip files by itself.

**Before you run**
- The next code cell loads the events.

In [ ]:
BASE = "/Volumes/workspace/default/bdcc"
from pyspark.sql import functions as F

events = spark.read.json(f"{BASE}/gharchive/*.json.gz")

Unlike pandas DataFrames and many database tables, Spark DataFrames allow `Struct` types, which create
**nested columns**. A **struct** is a column whose values hold named fields,
like a folder holding files and more folders. A struct inside a struct is a folder inside a folder.
`printSchema()` shows the nesting with indentation. For these events the schema is very long (over 700 lines).
Scroll through it. Look only at the lines that start with ` |-- `: those are the top-level columns.

In [ ]:
events.printSchema()

That is a lot. Two shorter views help: `columns` lists only the top level,
and `select(...).printSchema()` shows only the columns you pick.

In [ ]:
print(events.count())
events.columns

In [ ]:
events.select("actor", "repo", "type").printSchema()

`actor` (who did it) and `repo` (where) are structs. `type` is the kind of event.

## Dot paths

Refer to a nested field with a dot: `"actor.login"` is the `login` field inside `actor`.
Original cell 140, ported to the events:

In [ ]:
events.select("actor.login", "repo.name").limit(10).show()

The result columns are called `login` and `name`: the last part of each path.

## Broken on purpose: a field without its folder

The next cell asks for `login` without saying where it lives. Predict the error.

In [ ]:
# EXPECT-ERROR: AnalysisException
# Broken on purpose: login lives inside actor.
events.select("login").show()

**What you should see** (real output, trimmed):

```text
Traceback (most recent call last):
  File "<cell 105>", line 3, in <module>
    events.select("login").show()
  ...
pyspark.errors.exceptions.connect.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `login` cannot be resolved. Did you mean one of the following? [`org`, `id`, `repo`, `type`, `actor`]. SQLSTATE: 42703;
'Project ['login]
```

Spark looks for `login` only at the top level. The "Did you mean" list offers top-level columns.
Two of them hold a field called `login`: `org` and `actor`.
`org` holds the login of an organization. `actor` holds the login of the person who did the event.
So pick the folder by meaning: here it is `actor`.
To see what a folder holds, run `events.select("actor").printSchema()`.

**Fix it.** Give the full path.

In [ ]:
events.select("actor.login").show(5)

## Dot paths everywhere

Dot paths also work in `filter`, `groupBy` and `orderBy`. `F.col("repo.name")` is the Column version.
`.alias(...)` gives a result column a clearer name.

How many events of each type are there?

In [ ]:
events.groupBy("type").count().orderBy(F.desc("count"), "type").show()

Starring a repository creates a `WatchEvent`. A `ForkEvent` means someone copied a repository to their own account.
Here are a few forks, with clearer column names:

In [ ]:
(
    events.filter(F.col("type") == "ForkEvent")
    .select(F.col("actor.login").alias("user"), F.col("repo.name").alias("repo_name"))
    .show(5, truncate=False)
)

## Problem 5 (graded)

Create a function `starriest` that accepts the Spark DataFrame `df_events` and returns a Spark DataFrame with columns
`repo_name` and `stars`. `stars` is the number of `WatchEvent`s a repository received.
Keep the ten repositories with the most stars. Sort by decreasing `stars`, then by `repo_name`.

**The single-chain rule is relaxed: you may use several statements.**

Ties matter here: the sort by `repo_name` decides which repositories make the top ten.

In [ ]:
def starriest(df_events):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
from numpy.testing import assert_equal
slist = starriest(events).toPandas()
assert_equal(slist.columns.tolist(), ["repo_name", "stars"])
assert_equal(len(slist), 10)
assert_equal(
    slist.iloc[:5].to_numpy().tolist(),
    [
        ["cachethq/Cachet", 37],
        ["prakhar1989/awesome-courses", 34],
        ["wasabeef/awesome-android-ui", 26],
        ["begriffs/postgrest", 20],
        ["vinta/awesome-python", 18],
    ],
)
print("Problem 5 test passed")

## Discovery: where do the commits live?

A `PushEvent` sends one or more **commits** (saved changes to code) to a repository.
Your task: find where the commits live, and a tool that turns them into rows. Answer two questions:
1. What is the full path of the commits list, and what does each commit contain?
   Scroll through the long `printSchema()` output above and look for `commits`.
2. Which `F` tool gives each item of a list its own row?
   The next cell lists the names in the `F` toolbox that contain "explode". Read the manual of the one that fits.

In [ ]:
# Which names in the F toolbox contain "explode"?
[name for name in dir(F) if "explode" in name]

**Your turn.** Read the manual of the tool you picked with `help(...)`.
Then make a DataFrame with one row per commit, look at its schema, and count the rows.

In [ ]:
# Your code here: help(...), then one row per commit, printSchema() and count()

## Working with text files

Spark can also read plain text. Each line of the file becomes one row, in a column called `value`.
We use 23 public-domain books from Project Gutenberg (original cells 150 to 163).
The pattern `1/1/1/*/*/?????.txt` matches the book files: `*` matches any folder name, and `?` matches one character.

In [ ]:
texts = spark.read.text(f"{BASE}/gutenberg/1/1/1/*/*/?????.txt")
texts.show()

We count the number of lines in all the books (original cell 155).

In [ ]:
texts.count()

Original cell 157 lists the most frequent **tokens** (any run of characters without spaces):
- `lower` makes the text lower case, and `split(..., r"\s")` cuts each line at every whitespace character;
- `explode` gives each token its own row, and `filter(col("token") != "")` drops empty tokens;
- `groupby`, `count` and `orderBy(desc(...))` rank the tokens.

The cell keeps its original import line, `from pyspark.sql.functions import split, count, lower, col`.
That line does not import `explode` and `desc`: the original relied on earlier cells.
We add a second import line, so the cell also works on its own (remember Session 5).

In [ ]:
from pyspark.sql.functions import split, count, lower, col
from pyspark.sql.functions import explode, desc  # added: the original relied on an earlier import

(
    texts.select(explode(split(lower(texts.value), r"\s")).alias("token"))
    .filter(col("token") != "")
    .groupby("token")
    .count()
    .orderBy(desc("count"))
    .show()
)

To read each file as one single row, set `wholetext=True` (original cell 159).
Be careful: each book must then fit in the memory of one executor.

In [ ]:
(
    spark.read.text(
        f"{BASE}/gutenberg/1/1/1/*/*/?????.txt", wholetext=True
    ).show()
)

## Which file did a row come from?

The original notebook used the function `input_file_name()` here (original cells 161 to 163).
Databricks' data storage set-up, called Unity Catalog, does not support it. Instead, every file-based DataFrame has a hidden column,
`_metadata`. Its field `_metadata.file_path` holds the path of the file each row came from.

In [ ]:
(
    spark.read.text(
        f"{BASE}/gutenberg/1/1/1/*/*/?????.txt", wholetext=True
    )
    .withColumn("filename", F.col("_metadata.file_path"))
    .show()
)

In [ ]:
(
    spark.read.text(
        f"{BASE}/gutenberg/1/1/1/*/*/?????.txt", wholetext=True
    )
    .select("value", F.col("_metadata.file_path").alias("filename"))
    .show()
)

Counting per file shows how many rows each file gave. Here are the events per hour file:

In [ ]:
(
    events.groupBy(F.col("_metadata.file_path").alias("file"))
    .count()
    .orderBy("file")
    .show(truncate=False)
)

On Databricks the paths start with `dbfs:/Volumes/workspace/default/bdcc/`.

## The capstone

Open `spark-capstone.ipynb`. It is a teammate's notebook with four bugs. Find and fix them all.

## Session 6 check

Answer in your own words. Then check yourself in the DF Formats module.
In Intuition, use the "Your turn" at the end of the scene "Folders inside folders".
In Math & Code, use the "Your turn" questions under "Reading JSON and dot paths"
and "Problem 5 (graded): ranking with a tie-break".

1. Write the `select` that shows the GitHub user name stored inside `actor` and the repository name stored inside `repo`.
2. `events.select("login").show()` fails, and Spark suggests `org`, `id`, `repo`, `type` and `actor`.
   You want the user name of whoever did each event. How do you fix the cell, and why that suggestion?
3. On the full data, `Pathgather/please-wait` and `alvarotrigo/fullPage.js` both have 9 stars, at places 13 and 14.
   A ranking sorts by `F.desc("stars")`, then by `"repo_name"`, and keeps 13 rows with `limit(13)`.
   Which of the two is left out, and why?

# Optional stretch tasks

These problems are not graded. They use everything from Sessions 1 to 6.
In each one, **the single-chain rule is relaxed: you may use several statements.**

Three specs contain a deliberate mismatch, kept from the original notebook as a reading exercise:
**one name in the spec disagrees with its test. Run the test and read its error to find which.**

**Before you run**
- The next code cell loads everything the stretch tasks need.

In [ ]:
BASE = "/Volumes/workspace/default/bdcc"
from pyspark.sql import functions as F
from numpy.testing import assert_equal, assert_array_almost_equal

trips = spark.read.parquet(f"{BASE}/nyctaxi/green_2017")
events = spark.read.json(f"{BASE}/gharchive/*.json.gz")
texts = spark.read.text(f"{BASE}/gutenberg/1/1/1/*/*/?????.txt")

## Problem 4

Create a function `pickup` that accepts `trips` and returns a Spark DataFrame with columns `pickup_hour` and
`pickup_location`. The column `pickup_hour` is of `StringType` and corresponds to `lpep_pickup_datetime` rounded
down to the nearest hour, and the column `pickup_location` is the array of unique `PULocationID` for that
`pickup_hour` sorted by `PULocationID`. Sort the resulting data frame by `pickup_hour`.

**One name in this spec disagrees with its test. Run the test and read its error to find which.**

In [ ]:
def pickup(trips):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
pdf_pickup = pickup(trips).toPandas()
assert_equal(len(pdf_pickup), 2286)
assert_equal(pdf_pickup.columns.tolist(), ["pickup_hour", "pickup_locations"])
assert_equal(
    pdf_pickup.iloc[:10].to_numpy().tolist(),
    [
        ["2008-12-31 18:00:00", [37]],
        ["2008-12-31 23:00:00", [95, 181, 264]],
        ["2009-01-01 00:00:00", [95, 181, 193, 260]],
        ["2009-01-01 01:00:00", [7, 260]],
        ["2009-01-01 02:00:00", [260]],
        ["2009-01-01 05:00:00", [181]],
        ["2009-01-01 07:00:00", [181]],
        ["2009-01-01 08:00:00", [181]],
        ["2009-01-01 09:00:00", [85, 181]],
        ["2009-01-01 10:00:00", [181]],
    ],
)
print("Problem 4 test passed")

## Problem 6

A `PushEvent` carries its commits in `payload.commits` (see the discovery task in Session 6).
Create a function `top_committers` that accepts `df_events` and returns a Spark DataFrame with columns `login` and
`commits`: the number of commits pushed by each GitHub account (`actor.login`).
Keep the ten accounts with the most commits. Sort by decreasing `commits`, then by `login`.

In [ ]:
def top_committers(df_events):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
clist = top_committers(events).toPandas()
assert_equal(clist.columns.tolist(), ["login", "commits"])
assert_equal(len(clist), 10)
assert_equal(
    clist.iloc[:5].to_numpy().tolist(),
    [
        ["mirror-updates", 1239],
        ["KenanSulayman", 239],
        ["piwi", 190],
        ["nyov", 139],
        ["apmckinlay", 100],
    ],
)
print("Problem 6 test passed")

## Problem 7

Create a function `keyword_events` that accepts `df_events` and a `keyword`. It keeps the events whose repository
name (`repo.name`) contains `keyword` as a whole word, ignoring upper and lower case. "Whole word" means the
keyword is bounded by non-word characters, such as `/`, `-` or the end of the name.
For example, `python` matches `vinta/awesome-python` but not `pythonista`.
The function saves those events as Parquet in `f"{BASE}/output/keyword-events"`, partitioned by `type`.
It returns nothing.

Hint: look at `help(F.col("x").rlike)`. A **regular expression** is a text pattern:
`(?i)` ignores case, and `\b` marks a word boundary.

In [ ]:
def keyword_events(df_events, keyword):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
out_dir = f"{BASE}/output/keyword-events"
dbutils.fs.rm(out_dir, True)  # start from an empty folder (the original used !rm -rf)

keyword_events(events, "python")
folders = [f.name for f in dbutils.fs.ls(out_dir) if f.name.startswith("type=")]
assert_equal(len(folders), 11)
for name in folders:
    files = [f.name for f in dbutils.fs.ls(f"{out_dir}/{name}")]
    assert_equal(any(fn.endswith(".parquet") for fn in files), True)
df_saved = spark.read.parquet(out_dir)
assert_equal(df_saved.count(), 152)
assert_equal(set(df_saved.columns), set(events.columns))
print("Problem 7 test passed")

## Problem 8

Create a function `count_tokens` that accepts a list of Project Gutenberg filepaths `fpaths` and returns a Spark
DataFrame with columns `id` and `tokens` sorted by `id`. The `id` column corresponds to the id of the book and the
`value` column corresponds to the number of tokens in each book, including headers and footers.
A token is defined as a sequence of non-whitespace characters.

The book id is the number in the file name, for example `11102` in `.../11102.txt`.
Use `_metadata.file_path` (Session 6) to get the file name.

**One name in this spec disagrees with its test. Run the test and read its error to find which.**
(There is also a second, harmless difference between the spec and the test. Can you spot it?)

In [ ]:
def count_tokens(fpaths):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
pdf_token_counts = count_tokens(
    f"{BASE}/gutenberg/1/1/1/*/*/?????.txt"
).toPandas()
assert_equal(pdf_token_counts.columns.tolist(), ["id", "tokens"])
assert_equal(
    pdf_token_counts.iloc[:10].to_numpy().tolist(),
    [
        ["11102", 44876],
        ["11115", 78513],
        ["11129", 12947],
        ["11141", 64997],
        ["11144", 46242],
        ["11148", 11194],
        ["11149", 7695],
        ["11150", 13419],
        ["11160", 117798],
        ["11162", 4968],
    ],
)
print("Problem 8 test passed")

## Problem 9

Create a function `count_love` that reads a Spark DataFrame `df_texts` of strings and returns the number of
occurrences of the word `love` (case-insensitive, bounded by non-word characters) in all of the strings.

**One name in this spec disagrees with its test. Run the test and read its error to find which.**

In [ ]:
def count_love(texts):
    # YOUR CODE HERE
    raise NotImplementedError()

In [ ]:
love_count = count_love(texts)
assert_equal(type(love_count), int)
assert_equal(love_count, 241)
print("Problem 9 test passed")

**You reached the end.** Well done. Keep this notebook: the patterns here cover most everyday Spark work.